# 02 — Analytics & ML Modeling
## NASA C-MAPSS FD001 — Remaining Useful Life (RUL)

This notebook is the **Analytics + Machine Learning** stage of the project.

The previous notebook (`01_data_ingestion`) prepared the FD001 data and saved these Delta tables:

- `workspace.default.silver_cmapss_train`
- `workspace.default.silver_cmapss_test`
- `workspace.default.silver_cmapss_rul`

**Goal:** understand engine degradation, identify useful features, train regression models to predict Remaining Useful Life (RUL), compare the models, evaluate the final model on the official FD001 test set, and save outputs for the Visualization stage.

> **Team workflow:** This notebook is intended to be created in the same Databricks Git folder as the ingestion notebook. Since your team is keeping the project simple, everyone can work on `main`, but each member should work in their own notebook/file and **Pull before starting and Commit & Push after finishing**.


## 1. Load the prepared Delta tables

We reuse the prepared tables instead of reading and cleaning the raw files again.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Location of the uploaded Silver CSV files
VOLUME_PATH = "/Volumes/workspace/default/nyc_data_vol"

# Load Silver-layer CSV files
train_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/silver_cmapss_train.csv")
)

test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/silver_cmapss_test.csv")
)

rul_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/silver_cmapss_rul.csv")
)

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())
print("RUL rows:", rul_df.count())

train_df.printSchema()

## 2. Dataset overview

The training data contains engine ID, operating cycle, 3 operating settings, 21 sensor measurements, and the derived `RUL` target.


In [0]:
display(train_df.limit(10))
display(train_df.describe())


In [0]:
print("Number of training engines:", train_df.select("unit_number").distinct().count())
print("Number of test engines:", test_df.select("unit_number").distinct().count())
print("Training columns:", len(train_df.columns))
print("Test columns:", len(test_df.columns))


## 3. RUL analytics

RUL is the prediction target. We inspect its statistical distribution and how long each engine operated in the training set.


In [0]:
train_df.select("RUL").summary().show()


In [0]:
engine_life = (
    train_df
    .groupBy("unit_number")
    .agg(F.max("cycle").alias("lifetime_cycles"))
    .orderBy("unit_number")
)

display(engine_life)
display(engine_life.select("lifetime_cycles").summary())


In [0]:
# RUL progression for a few example engines.
selected_engines = [1, 10, 20, 50]

rul_progression = (
    train_df
    .filter(F.col("unit_number").isin(selected_engines))
    .select("unit_number", "cycle", "RUL")
    .orderBy("unit_number", "cycle")
)

display(rul_progression)


## 4. Sensor correlation analysis

We measure the Pearson correlation between each operating feature/sensor and RUL.

A large absolute correlation indicates a stronger **linear association** with RUL. Correlation alone does not prove causation or determine final model usefulness.


In [0]:
sensor_columns = [c for c in train_df.columns if c.startswith("sensor_")]
operating_columns = ["op_setting_1", "op_setting_2", "op_setting_3"]

correlation_rows = []

for feature in operating_columns + sensor_columns:
    corr_value = train_df.stat.corr(feature, "RUL")
    correlation_rows.append(
        (feature, float(corr_value) if corr_value is not None else None)
    )

correlation_df = spark.createDataFrame(
    correlation_rows,
    ["feature", "correlation_with_RUL"]
)

display(
    correlation_df
    .orderBy(F.abs(F.col("correlation_with_RUL")).desc())
)


## 5. Sensor variability analysis

We inspect the mean and standard deviation of the sensor/operating features. Very low-variance features may carry little information, but we do not remove features solely on this test.


In [0]:
feature_columns_for_stats = operating_columns + sensor_columns

stats_rows = []

for feature in feature_columns_for_stats:
    row = (
        train_df
        .select(
            F.mean(feature).alias("mean"),
            F.stddev(feature).alias("stddev")
        )
        .collect()[0]
    )

    stats_rows.append((feature, row["mean"], row["stddev"]))

feature_stats_df = spark.createDataFrame(
    stats_rows,
    ["feature", "mean", "stddev"]
)

display(feature_stats_df.orderBy("stddev"))


## 6. Prepare ML features

- `RUL` is the target and must not be used as an input feature.
- `unit_number` is an engine identifier and is excluded.
- `cycle` is retained because operating age is relevant to degradation.
- The three operating settings and 21 sensor measurements are included initially.


In [0]:
target_column = "RUL"

feature_columns = (
    ["cycle"]
    + operating_columns
    + sensor_columns
)

print("Number of model features:", len(feature_columns))
print(feature_columns)


## 7. Engine-level train/validation split

We split **whole engines**, not individual rows.

This avoids putting observations from the same engine into both training and validation sets, which can make a model look better than it really is.

The split is deterministic: 80% of engines for training and 20% for validation, with a fixed random seed.


In [0]:
import random

engine_ids = [
    row["unit_number"]
    for row in train_df
    .select("unit_number")
    .distinct()
    .collect()
]

engine_ids = sorted(engine_ids)
random.Random(42).shuffle(engine_ids)

split_index = int(len(engine_ids) * 0.8)

train_engine_ids = engine_ids[:split_index]
val_engine_ids = engine_ids[split_index:]

ml_train = train_df.filter(F.col("unit_number").isin(train_engine_ids))
ml_val = train_df.filter(F.col("unit_number").isin(val_engine_ids))

print("Training engines:", len(train_engine_ids))
print("Validation engines:", len(val_engine_ids))
print("Training rows:", ml_train.count())
print("Validation rows:", ml_val.count())


## 8. Assemble ML features

Spark ML models expect predictors in a single vector column called `features`.


In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_vector = assembler.transform(ml_train).select(
    "unit_number", "cycle", "RUL", "features"
)

val_vector = assembler.transform(ml_val).select(
    "unit_number", "cycle", "RUL", "features"
)

display(train_vector.limit(5))


# 9. Model 1 — Linear Regression

Linear Regression is used as a simple baseline.


In [0]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="RUL"
)

lr_model = lr.fit(train_vector)
lr_predictions = lr_model.transform(val_vector)

display(
    lr_predictions.select(
        "unit_number", "cycle", "RUL", "prediction"
    ).limit(20)
)


# 10. Model 2 — Random Forest Regression

Random Forest can model nonlinear relationships and interactions between measurements.


In [0]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="RUL",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf.fit(train_vector)
rf_predictions = rf_model.transform(val_vector)

display(
    rf_predictions.select(
        "unit_number", "cycle", "RUL", "prediction"
    ).limit(20)
)


# 11. Model 3 — Gradient-Boosted Trees

Gradient-Boosted Trees provide another nonlinear regression model for comparison.


In [0]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="RUL",
    maxIter=50,
    maxDepth=5,
    seed=42
)

gbt_model = gbt.fit(train_vector)
gbt_predictions = gbt_model.transform(val_vector)

display(
    gbt_predictions.select(
        "unit_number", "cycle", "RUL", "prediction"
    ).limit(20)
)


## 12. Compare validation performance

Metrics:

- **RMSE:** lower is better; larger errors receive more weight.
- **MAE:** lower is better.
- **R²:** higher is better.

All three models are evaluated on the same engine-level validation set.


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_evaluator = RegressionEvaluator(
    labelCol="RUL",
    predictionCol="prediction",
    metricName="rmse"
)

mae_evaluator = RegressionEvaluator(
    labelCol="RUL",
    predictionCol="prediction",
    metricName="mae"
)

r2_evaluator = RegressionEvaluator(
    labelCol="RUL",
    predictionCol="prediction",
    metricName="r2"
)

model_results = []

for model_name, predictions in [
    ("Linear Regression", lr_predictions),
    ("Random Forest", rf_predictions),
    ("Gradient Boosting", gbt_predictions)
]:
    model_results.append((
        model_name,
        float(rmse_evaluator.evaluate(predictions)),
        float(mae_evaluator.evaluate(predictions)),
        float(r2_evaluator.evaluate(predictions))
    ))

model_results_df = spark.createDataFrame(
    model_results,
    ["model", "RMSE", "MAE", "R2"]
)

display(model_results_df.orderBy("RMSE"))


## 13. Select the best model

For this project, validation RMSE is the primary ranking metric. MAE and R² remain available for interpretation.


In [0]:
best_model_row = (
    model_results_df
    .orderBy(F.col("RMSE").asc())
    .first()
)

best_model_name = best_model_row["model"]

print("Best model by validation RMSE:", best_model_name)
print("Validation RMSE:", best_model_row["RMSE"])
print("Validation MAE:", best_model_row["MAE"])
print("Validation R²:", best_model_row["R2"])


## 14. Random Forest feature importance

Feature importance helps identify which measurements the Random Forest relied on most.

This is **model-specific importance**, not causal importance.


In [0]:
importance_vector = rf_model.featureImportances

importance_rows = [
    (feature_columns[i], float(importance_vector[i]))
    for i in range(len(feature_columns))
]

feature_importance_df = (
    spark.createDataFrame(
        importance_rows,
        ["feature", "importance"]
    )
    .orderBy(F.col("importance").desc())
)

display(feature_importance_df)


## 15. Prepare the official FD001 test-set ground truth

The FD001 test file contains only the observed portion of each engine's lifecycle.

`RUL_FD001.txt` supplies the remaining cycles **after the final observed test cycle** for each engine.

Therefore:

`true_RUL(row) = terminal_RUL + (last_observed_cycle - current_cycle)`

For the final observed row of an engine, this becomes simply `terminal_RUL`.


In [0]:


rul_with_id = (
    rul_df
    .select(F.col("RUL").cast("int").alias("terminal_RUL"))
    .withColumn(
        "unit_number",
        F.row_number().over(
            Window.orderBy(F.monotonically_increasing_id())
        )
    )
    .select("unit_number", "terminal_RUL")
)

display(rul_with_id.limit(10))
print("RUL engines:", rul_with_id.count())

In [0]:
test_last_cycle = (
    test_df
    .groupBy("unit_number")
    .agg(F.max("cycle").alias("last_cycle"))
)

test_truth_df = (
    test_df
    .join(test_last_cycle, on="unit_number", how="left")
    .join(rul_with_id, on="unit_number", how="left")
    .withColumn(
        "true_RUL",
        F.col("terminal_RUL")
        + (F.col("last_cycle") - F.col("cycle"))
    )
)

print("Rows with missing terminal RUL:",
      test_truth_df.filter(F.col("terminal_RUL").isNull()).count())

display(
    test_truth_df.select(
        "unit_number",
        "cycle",
        "last_cycle",
        "terminal_RUL",
        "true_RUL"
    ).limit(20)
)


## 16. Retrain the candidate models on all labeled training data

After model comparison, the candidate models are retrained using all available labeled training engines.

This avoids discarding the validation engines when fitting the final candidate models.


In [0]:
full_train_vector = assembler.transform(train_df).select(
    "unit_number", "cycle", "RUL", "features"
)

final_lr_model = lr.fit(full_train_vector)
final_rf_model = rf.fit(full_train_vector)
final_gbt_model = gbt.fit(full_train_vector)


## 17. Generate predictions for the test set

We generate predictions from all three final candidate models. We keep the true RUL alongside the predictions so the results can be evaluated and passed to the Visualization stage.


In [0]:
test_vector = assembler.transform(test_truth_df)

lr_test = (
    final_lr_model.transform(test_vector)
    .select("unit_number", "cycle", "true_RUL", "prediction")
    .withColumnRenamed("prediction", "lr_prediction")
)

rf_test = (
    final_rf_model.transform(test_vector)
    .select("unit_number", "cycle", "prediction")
    .withColumnRenamed("prediction", "rf_prediction")
)

gbt_test = (
    final_gbt_model.transform(test_vector)
    .select("unit_number", "cycle", "prediction")
    .withColumnRenamed("prediction", "gbt_prediction")
)

test_predictions = (
    lr_test
    .join(rf_test, ["unit_number", "cycle"], "inner")
    .join(gbt_test, ["unit_number", "cycle"], "inner")
)

display(test_predictions.limit(20))


## 18. Evaluate the official test point for each engine

For C-MAPSS FD001, the supplied RUL values correspond to the **last observed cycle of each test engine**.

We therefore select the final observed cycle for each engine before calculating the test metrics.


In [0]:
last_cycle_window = Window.partitionBy("unit_number").orderBy(F.col("cycle").desc())

terminal_predictions = (
    test_predictions
    .withColumn("row_num", F.row_number().over(last_cycle_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

display(terminal_predictions.orderBy("unit_number"))
print("Terminal test engines:", terminal_predictions.count())


In [0]:
def evaluate_prediction_column(df, prediction_col):
    evaluator_rmse = RegressionEvaluator(
        labelCol="true_RUL",
        predictionCol=prediction_col,
        metricName="rmse"
    )
    evaluator_mae = RegressionEvaluator(
        labelCol="true_RUL",
        predictionCol=prediction_col,
        metricName="mae"
    )
    evaluator_r2 = RegressionEvaluator(
        labelCol="true_RUL",
        predictionCol=prediction_col,
        metricName="r2"
    )

    return (
        float(evaluator_rmse.evaluate(df)),
        float(evaluator_mae.evaluate(df)),
        float(evaluator_r2.evaluate(df))
    )

test_metric_rows = []

for model_name, prediction_col in [
    ("Linear Regression", "lr_prediction"),
    ("Random Forest", "rf_prediction"),
    ("Gradient Boosting", "gbt_prediction")
]:
    rmse_value, mae_value, r2_value = evaluate_prediction_column(
        terminal_predictions,
        prediction_col
    )

    test_metric_rows.append(
        (model_name, rmse_value, mae_value, r2_value)
    )

test_results_df = spark.createDataFrame(
    test_metric_rows,
    ["model", "RMSE", "MAE", "R2"]
)

display(test_results_df.orderBy("RMSE"))


## 19. C-MAPSS-style asymmetric scoring

The C-MAPSS benchmark commonly uses an asymmetric scoring function where late predictions can be penalized differently from early predictions.

Here `error = prediction - true_RUL`.

- Negative error = predicted RUL is too low.
- Positive error = predicted RUL is too high.

The score below follows the common C-MAPSS scoring formulation and is **lower-is-better**.


In [0]:
from pyspark.sql.types import DoubleType
import math

@F.udf(returnType=DoubleType())
def cmapss_score(error):
    if error is None:
        return None

    if error < 0:
        return math.exp(-error / 13.0) - 1.0
    else:
        return math.exp(error / 10.0) - 1.0

scored_test = (
    terminal_predictions
    .withColumn("rf_error", F.col("rf_prediction") - F.col("true_RUL"))
    .withColumn("rf_cmapss_score", cmapss_score(F.col("rf_error")))
    .withColumn("lr_error", F.col("lr_prediction") - F.col("true_RUL"))
    .withColumn("lr_cmapss_score", cmapss_score(F.col("lr_error")))
    .withColumn("gbt_error", F.col("gbt_prediction") - F.col("true_RUL"))
    .withColumn("gbt_cmapss_score", cmapss_score(F.col("gbt_error")))
)

cmapss_results = spark.createDataFrame(
    [
        ("Linear Regression", float(scored_test.agg(F.sum("lr_cmapss_score")).first()[0])),
        ("Random Forest", float(scored_test.agg(F.sum("rf_cmapss_score")).first()[0])),
        ("Gradient Boosting", float(scored_test.agg(F.sum("gbt_cmapss_score")).first()[0]))
    ],
    ["model", "C_MAPSS_score"]
)

display(cmapss_results.orderBy("C_MAPSS_score"))


## 20. Select the final model

The final model is selected primarily from the **validation RMSE** comparison.

The test-set results are used to report final performance, not to tune the model.


In [0]:
if best_model_name == "Linear Regression":
    final_model = final_lr_model
    final_prediction_col = "lr_prediction"
elif best_model_name == "Random Forest":
    final_model = final_rf_model
    final_prediction_col = "rf_prediction"
else:
    final_model = final_gbt_model
    final_prediction_col = "gbt_prediction"

print("Final selected model:", best_model_name)


## 21. Create final outputs for the Visualization stage

These Delta tables are the hand-off between the ML stage and the Visualization stage.


In [0]:
# 1) Model comparison
model_results_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.ml_model_results"
)

# 2) Random Forest feature importance
feature_importance_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.ml_feature_importance"
)

# 3) Validation predictions from all models
validation_predictions = (
    lr_predictions
    .select("unit_number", "cycle", "RUL", "prediction")
    .withColumnRenamed("prediction", "lr_prediction")
    .join(
        rf_predictions
        .select("unit_number", "cycle", "prediction")
        .withColumnRenamed("prediction", "rf_prediction"),
        ["unit_number", "cycle"],
        "inner"
    )
    .join(
        gbt_predictions
        .select("unit_number", "cycle", "prediction")
        .withColumnRenamed("prediction", "gbt_prediction"),
        ["unit_number", "cycle"],
        "inner"
    )
)

validation_predictions.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.ml_validation_predictions"
)

# 4) Final test predictions
final_test_output = (
    test_predictions
    .withColumn("selected_model", F.lit(best_model_name))
    .withColumn("final_prediction", F.col(final_prediction_col))
)

final_test_output.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.ml_test_predictions"
)

print("Saved ML output tables successfully.")


## 22. Verify the hand-off tables

The Visualization teammate can read these tables directly from Databricks.


In [0]:
display(spark.table("workspace.default.ml_model_results"))
display(spark.table("workspace.default.ml_feature_importance"))
display(spark.table("workspace.default.ml_test_predictions").limit(20))


# Final checklist

- [ ] All cells run successfully.
- [ ] Validation is performed at the engine level.
- [ ] All three models have RMSE, MAE and R² results.
- [ ] Feature importance is generated.
- [ ] Official FD001 test RUL is reconstructed correctly.
- [ ] Terminal-cycle test metrics are calculated.
- [ ] Final model is selected based on validation performance.
- [ ] ML output Delta tables are saved.
- [ ] Commit and push the notebook to GitHub.

### Output tables for the Visualization teammate

```text
workspace.default.ml_model_results
workspace.default.ml_feature_importance
workspace.default.ml_validation_predictions
workspace.default.ml_test_predictions
```
